In [0]:
print(dbutils.secrets.list("retail-platform-dev"))

[SecretMetadata(key='aws-access-key-id'), SecretMetadata(key='aws-secret-access-key'), SecretMetadata(key='cosmos-connection-string'), SecretMetadata(key='sqlserver-password'), SecretMetadata(key='sqlserver-username')]


In [0]:

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

In [0]:

from datetime import date
import json
from common_utils.logging import get_logger
from common_utils.ingestor import read_s3_parquet, write_raw

# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("cosmosdb_ingestion")

# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

# ============================================================
# 2. READ CONFIG
# ============================================================
with open(config_path, "r") as f:
    config = json.load(f)


# ============================================================
# 3. CONFIG SECTIONS
# ============================================================

source_config = config["source"]
target_config = config["target"]

# ============================================================
# 4. RUN DATE
# ============================================================
run_date = date.today().isoformat() 
logger.info("Fetching todays date %s", run_date)

aws_access_key_id = dbutils.secrets.get(scope="retail-platform-dev", key=source_config["aws_access_key_id"])
aws_secret_access_key = dbutils.secrets.get(scope="retail-platform-dev", key=source_config["aws_secret_access_key"])

# ============================================================
# 5. READ FROM COSMOSDB SERVER
# ============================================================
df = read_s3_parquet(spark, aws_access_key_id, aws_secret_access_key, source_config["path"])

logger.info("Data successfully read from  S3")


# ============================================================
# 6. TARGET PATH
target_path = f"{target_config['base_path']}/{target_config['folder']}/load_date={run_date}"

# ============================================================
# 7. WRITE RAW
# ============================================================
logger.info("Writing the in the target %s", target_path)

write_raw(df,target_path,target_config["file_format"],target_config["mode"])
logger.info("Data Succesfully written")

2026-10-28 11:54:23,964 INFO Fetching todays date 2026-10-28
2026-10-28 11:54:24,102 INFO Data successfully read from  S3
2026-10-28 11:54:24,103 INFO Writing the in the target /Volumes/retaildata_platform/bronze/raw_data//s3_sales/load_date=2026-10-28
2026-10-28 11:54:31,709 INFO Data Succesfully written
